# 品詞や原形のパターンで表現を抽出する (spaCy Matcher)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_ner/spacy_matcher.ipynb)

NLP クックブック 17-06 ／ 固有表現抽出

「形容詞 + 名詞」「〜することができる」のように、品詞や原形を使ったパターンに合う箇所を見つけたい。正規表現の単語版。

In [1]:
!pip install -q ginza ja_ginza
!python -m spacy download en_core_web_sm -q

In [2]:
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")
matcher = Matcher(nlp.vocab)
matcher.add("ADJ_NOUN", [[{"POS": "ADJ", "OP": "+"}, {"POS": "NOUN", "OP": "+"}]], greedy="LONGEST")
matcher.add("BE_GOING_TO", [[{"LEMMA": "be"}, {"LOWER": "going"}, {"LOWER": "to"}, {"POS": "VERB"}]])

doc = nlp("Large language models are going to change natural language processing. She was going to buy a new red car.")
for match_id, start, end in matcher(doc):
    print(nlp.vocab.strings[match_id], "|", doc[start:end].text)

BE_GOING_TO | are going to change
BE_GOING_TO | was going to buy
ADJ_NOUN | Large language models
ADJ_NOUN | natural language processing
ADJ_NOUN | new red car


In [3]:
# 日本語: 「〜ことができる」(可能表現) と「サ変名詞 + する」
nlp_ja = spacy.load("ja_ginza")
m = Matcher(nlp_ja.vocab)
m.add("KOTO_GA_DEKIRU", [[{"POS": "VERB"}, {"ORTH": "こと"}, {"ORTH": {"IN": ["が", "も", "は"]}}, {"LEMMA": "できる"}]])
m.add("SAHEN", [[{"POS": {"IN": ["NOUN", "VERB"]}, "TAG": {"REGEX": "サ変可能"}}, {"LEMMA": "する"}]])

doc = nlp_ja("このモデルは長い文章を要約することができる。日本語を翻訳することもできるし、質問に回答する。")
for match_id, start, end in m(doc):
    print(nlp_ja.vocab.strings[match_id], "|", doc[start:end].text)

SAHEN | 要約する
SAHEN | 翻訳する
SAHEN | 回答する


- 使える属性は `ORTH` `LOWER` `LEMMA` `POS` `TAG` `DEP` `ENT_TYPE` `IS_DIGIT` など。`OP` は `?` `*` `+` `!`。
- 単語列の辞書（数万語）を高速に引くなら `PhraseMatcher`、係り受けパターンなら `DependencyMatcher`。